# Capture activation traces

We collect attention and MLP projection inputs from dense forward passes
of Qwen2.5-0.5B-Instruct on three text prompts.

In [1]:
from contextlib   import ExitStack
from pathlib      import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

import json
import torch

/home/yabsed/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model

In [2]:
MODEL_ID       = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    revision = MODEL_REVISION
)

net = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    revision    = MODEL_REVISION, 
    torch_dtype = torch.bfloat16
).to(device).eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 325.73it/s, Materializing param=model.norm.weight]                              


## Channel importance

For a projection input $x \in \mathbb{R}^{B \times T \times N}$,
the importance of channel $i$ is

$$
v_i = \frac{1}{BT}
\sum_{b=1}^{B}\sum_{t=1}^{T}|x_{b,t,i}|.
$$

The mean is computed in the activation dtype, then converted to float32.
Channels retain their original order.

In [4]:
traces = []

def record(layer, inputs):

    n = layer. in_features 
    d = layer.out_features

    values = (
        inputs[0]
        .detach()
        .reshape(-1, n)

        .abs()
        .mean(dim=0)

        .float()
        .cpu()
        .tolist()
    )
    
    traces.append({
        "trace_id"  : len(traces),
        "n"         : n,
        "d"         : d,
        "row_bytes" : d * layer.weight.element_size(),
        "values"    : values
    })

In [5]:
PROMPTS = (
    "Explain why contiguous flash reads can be faster than scattered reads.",
    "Summarize the trade-off between neural-network sparsity and accuracy.",
    "Write a short Python function that computes a moving average.",
)

with ExitStack() as hooks:
    
    for name, layer in net.named_modules():
        if (
            isinstance(layer, torch.nn.Linear)
            and ".layers." in name
            and name.endswith("_proj")
        ):
            hooks.enter_context(
                layer.register_forward_pre_hook(record)
            )

    with torch.inference_mode():
        for input_index, prompt in enumerate(PROMPTS):
            
            text = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize              = False,
                add_generation_prompt = True,
            )

            tokens = tokenizer(
                text,
                return_tensors = "pt",
                truncation     = True,
                max_length     = 256,
            ).to(device)

            net(**tokens, use_cache=False)

            print(f"Prompt {input_index + 1} captured")


Prompt 1 captured
Prompt 2 captured
Prompt 3 captured


## Output

Save the importance vectors, projection dimensions, and weight-row sizes
to `activation_traces.json`.

In [6]:
TRACE = Path("activation_traces.json")

In [7]:
data = {
    "weight_bytes" : next(net.parameters()).element_size(),
    "traces"       : traces,
}

TRACE.write_text(json.dumps(data, indent=2))
len(traces)

504